In [2]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from itertools import *

def print_image(image):
    cv2.namedWindow("Sample Image", cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Sample Image', 800, 800)
    cv2.imshow('Sample Image', image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

shapeX = 0
shapeY = 0

def print_palette(colors):
    palette = np.zeros((100, 1500, 3), dtype=np.uint8)
    for i in range(len(colors)):
        for y in range(100):
            for x in range(int(1500//len(colors))):
                palette[y][i*int(1500//len(colors)) + x] = colors[i]
    
    print_image(palette)

step 1: set target image


input:
- result image size (shapeX, shapeY)

- boxCount (to optimize make priorty image in step 3)

- targetImageSrc

- targetImageType

- colorDiff for outline & makepalette

output
- outlined image

In [3]:
shapeY = 300
shapeX = 300
boxCount = 50


targetImageSrc = "jjowayo"
targetImageType = ".JPG"
diff = 40



targetImage = cv2.resize(cv2.imread('images/' + targetImageSrc + targetImageType, cv2.IMREAD_COLOR), (shapeX,shapeY),interpolation = cv2.INTER_NEAREST)

outlines = []
boxRangeX = shapeX/boxCount
boxRangeY = shapeY/boxCount
targetOutlineBox = np.zeros((boxCount, boxCount, 0)).tolist()
outlineOnlyTargetImage = np.zeros((shapeY,shapeX,3), dtype=np.uint8)
outlinedTargetImage = targetImage.copy()

for y in range(0, shapeY-1):
    for x in range(0, shapeX-1):
        if(abs(np.linalg.norm(cv2.add(targetImage[y][x], -1*targetImage[y+1][x]))) >= diff or 
           abs(np.linalg.norm(cv2.add(targetImage[y][x], -1*targetImage[y][x+1]))) >= diff):
            outlines.append([y,x])
            outlineOnlyTargetImage[y][x] = [255,0,0]
            outlinedTargetImage[y][x] = [0,0,0]
            targetOutlineBox[int(y//boxRangeY)][int(x//boxRangeX)].append([y,x])


print_image(outlinedTargetImage)
print_image(outlineOnlyTargetImage)

step 2: generate priorty image, steps

input:

output: (targetImage)
- priorty image (min dst from outline)
- steps (each steps sorted by min dst from outline)

In [4]:
boxCheck = []
checkDirections = [[1,0],[1,1],[1,-1],[0,-1],[-1,-1],[-1,0],[-1,1],[0,1]]

def checkBoxImages(boxX,boxY, outlineBox):
    newImage = np.zeros((shapeY, shapeX, 3), dtype=np.uint8)
    print(len(outlineBox[boxY][boxX]))
    for outlinePoint in outlineBox[boxY][boxX]:
        newImage[outlinePoint[0]][outlinePoint[1]] = [255,0,0]
    
    print_image(newImage)

def getMinDistanceStart(point, image, outlineBox):
    boxCheck.clear()
    dstP = getMinDistance(np.array(point), int(point[0]//boxRangeY), int(point[1]//boxRangeX), np.linalg.norm(np.array([shapeY,shapeX])), outlineBox)

    return dstP

def getMinDistance(point, boxY, boxX, minDstP, outlineBox):
    if([boxY, boxX] in boxCheck):
        return minDstP
    else:
        boxCheck.append([boxY, boxX])

    minDst = minDstP
    for outlinePoint in outlineBox[boxY][boxX]:
        dst = abs(np.linalg.norm(np.array(point) - np.array(outlinePoint)))
        if(dst < minDst):
            minDst = dst


    

    boxYO = boxCheck[0][0]
    boxXO = boxCheck[0][1]
        
    for direction in checkDirections:
        newBoxY = boxY + direction[0]
        newBoxX = boxX + direction[1]

        if(newBoxX < 0 or newBoxX >= boxCount or newBoxY < 0 or newBoxY >= boxCount):
            continue

        newDetY = 0
        newDetX = 0

        if(newBoxY > boxYO):
            newDetY = (boxY + 1) * boxRangeY
        elif(newBoxY == boxYO):
            newDetY = point[0]
        else:
            newDetY = boxY * boxRangeY
        
        if(newBoxX > boxXO):
            newDetX = (boxX + 1) * boxRangeX
        elif(newBoxX == boxXO):
            newDetX = point[1]
        else:
            newDetX = boxX * boxRangeX
        
        dst = abs(np.linalg.norm(np.array(point) - np.array([newDetY, newDetX])))

        if(dst < minDst):
            newDst = getMinDistance(point, newBoxY, newBoxX, minDst, outlineBox)
            if(newDst < minDst):
                minDst = newDst


    if(minDst < 1):
        return 1

    return minDst



def getPriortyImage(image, outlineBox):
    priortyImage = np.zeros((shapeY,shapeX,3), dtype=np.uint8)

    for y in range(0,shapeY):
        for x in range(0, shapeX):
            dstP = getMinDistanceStart([y,x], image, outlineBox)
            priortyImage[y][x] = [int(255//(dstP)), int(255//(dstP)), int(255//(dstP))]

    return priortyImage

def getSteps(priortyImage):
    stepPixels = {}
    steps = []

    for y in range(shapeY):
        for x in range(shapeX):
            if(str(priortyImage[y][x][0]) not in stepPixels):
                steps.append(str(priortyImage[y][x][0]))
                stepPixels[str(priortyImage[y][x][0])] = [[y,x]]
            else:
                stepPixels[str(priortyImage[y][x][0])].append([y,x])

    steps = sorted(steps, key = lambda x : int(x), reverse=True)

    return steps, stepPixels


targetPriortyImage = getPriortyImage(targetImage, targetOutlineBox)
targetSteps, targetStepPixels = getSteps(targetPriortyImage)
print("target priorty done")

target priorty done


step 3: get colorpalette, simplified image

input:
- outlineDst (outline to )

output: (targetImage)
- colorpalette 
- simplified image (use colors in colorpalette)

In [5]:
outlineDst = 2 #outlineDst 안의 것들은 그냥 가장 비슷한 색상으로 퉁친다


def getPalette(image, diffC, steps, stepPixels):
    palette = []
    paletteColors = []
    paletteColorsPixels = []
    simpleIndexImage = np.zeros((shapeY,shapeX))



    def getNearestColor(color, pixel, step):
        minDst = 442
        minDstIdx = -1
        for i, pColor in enumerate(palette):
            newDst = abs(np.linalg.norm(cv2.add(np.array(color) ,-1*np.array(pColor))))
            if(int(step) >= 255/outlineDst):
                if(newDst <= minDst):
                    minDst = newDst
                    minDstIdx = i
            else:
                if(newDst <= diffC):
                    paletteColors[i].append(np.array(color))
                    paletteColorsPixels[i].append(pixel)
                    palette[i] = cv2.add(np.array(palette[i])*(len(paletteColors[i])-1), np.array(color))/len(paletteColors[i])
                    return i
                
        
        if(int(step) >= 255/outlineDst and minDstIdx != -1):
            paletteColorsPixels[minDstIdx].append(pixel)
            return minDstIdx
        
        else:
            palette.append(color)
            paletteColors.append([color])
            paletteColorsPixels.append([pixel])
            return len(palette)-1
    


    for step in steps[::-1]:
        for y,x in stepPixels[step]:
            simpleIndexImage[y][x] = getNearestColor(image[y][x], [y,x], step)

    for i in range(len(palette)):
        color = palette[i]
        palette[i] = [int(color[0]), int(color[1]), int(color[2])]
    
    return palette, simpleIndexImage, paletteColorsPixels



def getSimpleImage(palette, simpleIndexImage):
    simpleImage = np.zeros((shapeY,shapeX,3), dtype=np.uint8)

    for y in range(0, shapeY):
        for x in range(0, shapeX):
            simpleImage[y][x] = palette[int(simpleIndexImage[y][x])]

    return simpleImage


targetPalette, simpleIndexTargetImage, targetPaletteColorsPixels = getPalette(outlinedTargetImage, diff, targetSteps, targetStepPixels)

simpleTargetImage = getSimpleImage(targetPalette, simpleIndexTargetImage)

# print_image(simpleTargetImage)
# print_palette(targetPalette)
print("target palette done")



def sortPaletteByVolume(paletteO, paletteColorsPixelsO):
    palette = paletteO.copy()
    paletteColorsPixels = paletteColorsPixelsO.copy()
    for i in range(len(palette) - 1):
        for j in range(i+1, len(palette)):
            if(len(paletteColorsPixels[i]) < len(paletteColorsPixels[j])):
                c = palette[i].copy()
                palette[i] = palette[j].copy()
                palette[j] = c.copy()

                l = paletteColorsPixels[i].copy()
                paletteColorsPixels[i] = paletteColorsPixels[j].copy()
                paletteColorsPixels[j] = l.copy()

    return palette, paletteColorsPixels


targetPalette, targetPaletteColorsPixels = sortPaletteByVolume(targetPalette, targetPaletteColorsPixels)

targetPaletteColorCount = [len(targetPaletteColorsPixels[i]) for i in range(len(targetPaletteColorsPixels))]

target palette done


step 4: get colorSheetImage

input:
- goalColorCount (goal colorSheetImage's palette color count)
- goalRange (goalColorCount range)

output:
- colorSheetImage (noise)
- colorSheetSteps
- colorSheetStepPixels
- colorSheetPalette
- colorSheetPaletteColorsPixels

In [23]:
colorSheetImage = np.zeros((shapeY,shapeX,3), dtype=np.uint8)
colorSheetSteps = ["1"]
colorSheetStepPixels = {"1" : []}

for y in range(0,shapeY):
    for x in range(0, shapeX):
        colorSheetImage[y][x] = [np.random.randint(0,255),np.random.randint(0,255),np.random.randint(0,255)]
        colorSheetStepPixels["1"].append([y,x])



goalColorCount = 20
goalRange = 5

colorSheetPalette = []
colorSheetPaletteColorsPixels = []
colorSheetDiff = 0


currentDiff = 0
upperDiff = 442
lowerDiff = 0
while(True):
    currentDiff = (lowerDiff + upperDiff)/2
    newColorPalette, _, _ = getPalette(colorSheetImage, currentDiff, colorSheetSteps, colorSheetStepPixels)
    colorCount = len(newColorPalette)

    if(abs(colorCount - goalColorCount) <= goalRange and colorCount >= goalColorCount):
        colorSheetDiff = currentDiff
        print("Done!!")
        print("goalColorCount: " + str(goalColorCount) + "    ColorCount: " + str(colorCount))
        print("Diff: " + str(colorSheetDiff))
        print()

        colorSheetPalette, _, colorSheetPaletteColorsPixels = getPalette(colorSheetImage, currentDiff, colorSheetSteps, colorSheetStepPixels)
        break

    elif(colorCount > goalColorCount):
        lowerDiff = currentDiff
    
    elif(colorCount < goalColorCount):
        upperDiff = currentDiff
    

    print("Diff: " + str(currentDiff) + "    ColorCount: " + str(colorCount))
    print("upperDiff: " + str(upperDiff) + "    lowerDiff: " + str(lowerDiff))
    print()


colorSheetPalette, colorSheetPaletteColorsPixels = sortPaletteByVolume(colorSheetPalette, colorSheetPaletteColorsPixels)

colorSheetPaletteColorCount = [len(colorSheetPaletteColorsPixels[i]) for i in range(len(colorSheetPaletteColorsPixels))]

Diff: 221.0    ColorCount: 1
upperDiff: 221.0    lowerDiff: 0

Diff: 110.5    ColorCount: 15
upperDiff: 110.5    lowerDiff: 0

Diff: 55.25    ColorCount: 105
upperDiff: 110.5    lowerDiff: 55.25

Diff: 82.875    ColorCount: 44
upperDiff: 110.5    lowerDiff: 82.875

Diff: 96.6875    ColorCount: 27
upperDiff: 110.5    lowerDiff: 96.6875

Done!!
goalColorCount: 20    ColorCount: 22
Diff: 103.59375



step 5: convert image

input:

output:
- result image

In [24]:
colorProportions = np.zeros((len(targetPalette), len(colorSheetPalette)))
colorProportionsPixelCount = np.zeros((len(targetPalette), len(colorSheetPalette)))
resultColorCount = 0
indices = np.arange(len(colorSheetPalette))
mergeColorCount = 1
colorSheetPaletteColorCountUsable = colorSheetPaletteColorCount.copy()
doneTargetColors = np.zeros((len(targetPalette)))

while(resultColorCount < len(targetPalette) and mergeColorCount <= len(colorSheetPalette)):
    mergeColors = list(combinations(indices, mergeColorCount))
    for i in range(len(mergeColors)):
        cnt = 0

        colors = mergeColors[i]

        doCheck = True

        for j in colors:
            if(colorSheetPaletteColorCountUsable[j] <= 0): doCheck = False
            else : cnt += colorSheetPaletteColorCountUsable[j]
        
        if(doCheck == False):
            continue

        colorMakeIdx = -1
        for k in range(len(targetPalette)):
            if(cnt >= targetPaletteColorCount[len(targetPalette) - 1 - k]):
                if(doneTargetColors[len(targetPalette) - 1 - k] != 1):
                    colorMakeIdx = len(targetPalette) - 1 - k
                else:
                    pass
            else:
                break
        
        
        if(colorMakeIdx != -1):
            colors = sorted(mergeColors[i], key = lambda x : colorSheetPaletteColorCountUsable[x])
            cnt = 0
            amount = -1
            amountPixel = -1

            for j in colors:
                doneTargetColors[colorMakeIdx] = 1

                if(targetPaletteColorCount[colorMakeIdx] - cnt > colorSheetPaletteColorCountUsable[j]):
                    colorProportions[colorMakeIdx][j] = round(colorSheetPaletteColorCountUsable[j] / targetPaletteColorCount[colorMakeIdx], 3)
                    colorProportionsPixelCount[colorMakeIdx][j] = colorSheetPaletteColorCountUsable[j]
                    cnt += colorSheetPaletteColorCountUsable[j]
                    colorSheetPaletteColorCountUsable[j] = 0
                else:
                    if((colors.index(j)) == len(colors) - 1):
                        colorProportionsPixelCount[colorMakeIdx][j] = targetPaletteColorCount[colorMakeIdx] - cnt
                        colorProportions[colorMakeIdx][j] = round(targetPaletteColorCount[colorMakeIdx] - cnt / targetPaletteColorCount[colorMakeIdx], 3)
                        colorSheetPaletteColorCountUsable[j] -= targetPaletteColorCount[colorMakeIdx] - cnt
                        
                    else:
                        if(amount < 0): 
                            amount = 1//(len(colors) - (colors.index(j))) * (targetPaletteColorCount[colorMakeIdx] - cnt)
                            amountPixel = amount
                            
                        colorProportions[colorMakeIdx][j] = round(amount / targetPaletteColorCount[colorMakeIdx], 3)
                        colorProportionsPixelCount[colorMakeIdx][j] = amount
                        cnt += amount
                        colorSheetPaletteColorCountUsable[j] -= amount
                
                if(colorSheetPaletteColorCountUsable[j] <= 0):
                    indices = np.delete(indices, list(indices).index(j))

            resultColorCount += 1
    
    print(mergeColorCount)
    mergeColorCount += 1
    
if(resultColorCount < len(targetPalette)):
    print("failed")


for i in range(len(colorSheetPalette)):
    colorSheetPaletteColorsPixels[i] = sorted(colorSheetPaletteColorsPixels[i], key = lambda x : (abs(np.linalg.norm(cv2.add(np.array(colorSheetPalette[i]), -1*np.array(x))))))


def getIndex(l, a):
    for i in range(len(l)):
        if(all(l[i] == a)):
            return i
    
    return 0

def getColor(proportionRange):

    colorIdx = -1

    rand = np.random.rand(1)

    for i in range(len(proportionRange)):
        if((rand <= proportionRange[i] and proportionRange[i] != 0)):
            colorIdx = i
            break

    return colorIdx

def getProportionRange(proportionPixelCount):
    proportionPixelCountRange = proportionPixelCount.copy()
    cnt = 0
    for i in range(len(proportionPixelCount)):
        if(proportionPixelCount[i] != 0):
            cnt += proportionPixelCount[i]
            proportionPixelCountRange[i] = cnt

    return np.array(proportionPixelCountRange) / cnt



resultImage = np.zeros((shapeY,shapeX,3), dtype=np.uint8)
resultPosImage = np.zeros((shapeY, shapeX, 2))
resultPImage = np.zeros((shapeY, shapeX, 2))

cntNull = 0 #예외케이스를 잡기 위한 카운트
colorProportionsPixelCountCopy = colorProportionsPixelCount.copy()
colorSheetPaletteColorsPixelsIdx = np.zeros((len(colorSheetPalette)), dtype=np.int64)

for step in targetSteps:
    for y,x in targetStepPixels[step]:
        pixelColor = simpleTargetImage[y][x]
        pixelColorIdx = getIndex(targetPalette, pixelColor)

        pixelColorProportionRange = getProportionRange(colorProportionsPixelCountCopy[pixelColorIdx])
        
        newPixelColorIdx = getColor(pixelColorProportionRange)

        if(newPixelColorIdx != -1):
            
            pixelPosition = colorSheetPaletteColorsPixels[newPixelColorIdx][colorSheetPaletteColorsPixelsIdx[newPixelColorIdx]]
            newPixelColor = colorSheetImage[pixelPosition[0]][pixelPosition[1]]
            resultImage[y][x] = newPixelColor
            resultPosImage[pixelPosition[0]][pixelPosition[1]] = [y,x]
            resultPImage[y][x] = [pixelPosition[0], pixelPosition[1]]


            colorSheetPaletteColorsPixelsIdx[newPixelColorIdx] += 1
            colorProportionsPixelCountCopy[pixelColorIdx][newPixelColorIdx] -= 1

        else:
            cntNull += 1
            resultImage[y][x] = [255,0,0]

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20


In [25]:
print_image(resultImage)

step 6: make animation

input:
- beforeChangeFrame
- changeFrame
- afterChangeFrame
- fps

output:
- delay beforeChangeFrame, change while changeFrame, remain afterChangeFrame

In [18]:
beforeChangeFrame = 30
changeFrame = 180
afterChangeFrame = 30
fps = 60
figsize = 8

import matplotlib.animation
import matplotlib.pyplot as plt
import numpy as np
fig = plt.figure(frameon=False)
fig.set_size_inches(figsize * 2 * (shapeX) / (shapeX + shapeY), figsize * 2 * (shapeY) / (shapeX + shapeY))
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 150
matplotlib.rcParams['animation.embed_limit'] = 2**128
matplotlib.rcParams['animation.ffmpeg_path'] = r'/opt/homebrew/Cellar/ffmpeg/8.0.1/bin/ffmpeg'
plt.ioff()

pltXoriginal = []
pltYoriginal = []
pltXtarget = []
pltYtarget = []
pltColor = []
pointPosImage = np.zeros((shapeY, shapeX, 2))
locationPosImage = np.zeros((shapeY, shapeX, 2))
undonePixels = []

for y in range(shapeY):
    for x in range(shapeX):
        pltXoriginal.append(x)
        pltYoriginal.append(shapeY - y)
        pltColor.append(np.array(colorSheetImage[y][x][::-1])/255)
        targetPos = resultPosImage[y][x]
        pltXtarget.append(targetPos[1])
        pltYtarget.append(shapeY - targetPos[0])
        pointPosImage[y][x] = [y,x]
        locationPosImage[y][x] = [y,x]
        undonePixels.append([y,x])

pltXoriginal = np.array(pltXoriginal)
pltYoriginal = np.array(pltYoriginal)
pltXtarget = np.array(pltXtarget)
pltYtarget = np.array(pltYtarget)

markersize = ((2 * fig.dpi * figsize /(shapeX + shapeY))/4)**2


pltXoriginalCopy = pltXoriginal.copy()
pltYoriginalCopy = pltYoriginal.copy()
pltXtargetCopy = pltXtarget.copy()
pltYtargetCopy = pltYtarget.copy()

frameImage = colorSheetImage.copy()


def changePixel(pixel):
    #ix,iy,locationPosY,locationPosX- 현 이미지의 좌표
    #pointPosY,pointPosX,ty,tx - 원 이미지의 좌표
    iy = pixel[0]
    ix = pixel[1]

    pointPosI = pointPosImage[iy][ix]
    pointPosIY = int(pointPosI[0])
    pointPosIX = int(pointPosI[1])
    

    ty = int(resultPosImage[pointPosIY][pointPosIX][0])
    tx = int(resultPosImage[pointPosIY][pointPosIX][1])

    pointPosT = pointPosImage[ty][tx]
    pointPosTY = int(pointPosT[0])
    pointPosTX = int(pointPosT[1])


    # 이 두 개를 바꿔! (iy,ix)가 (locY,locX)
    c = pltColor[iy * shapeY + ix].copy()
    pltColor[iy * shapeY + ix] = pltColor[ty * shapeY + tx].copy()
    pltColor[ty * shapeY + tx] = c.copy()

    c = frameImage[iy][ix].copy()
    frameImage[iy][ix] = frameImage[ty][tx].copy()
    frameImage[ty][tx] = c.copy()

    #정보 업데이트
    pointPosImage[iy][ix] = [pointPosTY,pointPosTX]
    locationPosImage[pointPosTY][pointPosTX] = [iy,ix]

    pointPosImage[ty][tx] = [pointPosIY, pointPosIX]
    locationPosImage[pointPosIY][pointPosIX] = [ty, tx]


    undonePixels.remove([ty,tx])



changingPixelsPerFrame = int(np.floor((shapeX*shapeY)/changeFrame))




def animate(t):
    f = changeFrame - beforeChangeFrame
    if(t - beforeChangeFrame < 0 or t > beforeChangeFrame + changeFrame):
        pass
    else:
        l = len(undonePixels)
        if(changingPixelsPerFrame > l):
            cnt = l
        else:
            cnt = changingPixelsPerFrame
        
        for i in range(cnt):
            idx = np.random.randint(0, l-i)
            pixel = undonePixels[idx]
            changePixel(pixel)

        plt.cla()
        plt.xlim(0,shapeX)
        plt.ylim(0,shapeY)
        plt.scatter(pltXoriginal, pltYoriginal, c = pltColor, marker='s', s = markersize)
        plt.axis('off')


plt.cla() 
plt.xlim(0,shapeX)
plt.ylim(0,shapeY)
plt.scatter(pltXoriginal, pltYoriginal, c = pltColor, marker='s', s = markersize)
plt.axis('off')
anim = matplotlib.animation.FuncAnimation(fig, animate, frames=beforeChangeFrame + changeFrame + afterChangeFrame, interval=1000/fps, repeat = False)


writer = matplotlib.animation.FFMpegFileWriter(fps = fps)
anim.save("videos/noise_to_" + targetImageSrc +".mp4", writer='ffmpeg')